In [ ]:
pip install -q transformers peft accelerate datasets scikit-learn

In [ ]:
!pip install -U "torchao==0.16.0"

## IMPORT

In [ ]:
import os
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from sklearn.metrics import f1_score

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## LOAD DATASET

In [ ]:
dataset = load_dataset("google/civil_comments")

In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])

## 7 LABELS

In [ ]:
LABELS = [
    "toxicity",
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
    "sexual_explicit"
]

NUM_LABELS = len(LABELS)

print("Number of labels:", NUM_LABELS)
print(LABELS)

## TEST DATASET

In [ ]:
train_ds = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(200_000))
)

val_ds = (
    dataset["validation"]
    .shuffle(seed=42)
    .select(range(20_000))
)

test_ds = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(20_000))
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

## LOAD BERTBETWEEN TOKENIZER

In [ ]:
MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    normalization=True
)

print("Tokenizer loaded")

## CREATING THE TOKENIZER FUNCTION

In [ ]:
def tokenize_function(examples):

    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokenized["labels"] = [
        [
            float(examples[label][i])
            for label in LABELS
        ]
        for i in range(len(examples["text"]))
    ]

    return tokenized

## TOKENIZE THE DATA

In [ ]:
train_tokenized = train_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=train_ds.column_names
)

val_tokenized = val_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=val_ds.column_names
)

test_tokenized = test_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=test_ds.column_names
)

In [ ]:
print(train_tokenized[0])

## LOAD BERTWEET FOR CLASSIFICATION

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print("Base model loaded")

## CREATE THE LoRA CONFIGURATION

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,

    r=8,

    lora_alpha=16,

    lora_dropout=0.1,

    target_modules=[
        "query",
        "value"
    ],

    modules_to_save=[
        "classifier"
    ],

    bias="none"
)

## ACTUALLY ATTACH LoRA

In [ ]:
lora_model = get_peft_model(
    base_model,
    lora_config
)

lora_model.print_trainable_parameters()

## CREATE THE METRICS

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probabilities = sigmoid(logits)

    predictions = (probabilities >= 0.5).astype(int)

    true_labels = (labels >= 0.5).astype(int)

    macro_f1 = f1_score(
        true_labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "macro_f1": macro_f1
    }

## CONFIGURE TRAINING

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    gradient_accumulation_steps=2,

    num_train_epochs=2,

    weight_decay=0.01,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_steps=100,

    report_to="none"
)

## CREATE THE TRAINER

In [ ]:
trainer = Trainer(
    model=lora_model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    compute_metrics=compute_metrics
)

## RUN THE SMOKE TEST|

In [ ]:
print("Starting LoRA training...")

trainer.train()

print("Training completed!")

## EVALUATE

In [ ]:
results = trainer.evaluate(test_tokenized)

print(results)

## MANUAL TEST

In [ ]:
texts = [
    "You are a wonderful person.",
    "You are an idiot.",
    "I will hurt you."
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
)

inputs = {
    k: v.to(trainer.model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = trainer.model(**inputs)

probabilities = torch.sigmoid(outputs.logits)

print(probabilities.cpu().numpy())

## MERGE LORA

In [ ]:
print("Merging LoRA adapters...")

merged_model = trainer.model.merge_and_unload()

print("Merge complete!")

In [ ]:
merged_model.config.id2label = {
    i: label
    for i, label in enumerate(LABELS)
}

merged_model.config.label2id = {
    label: i
    for i, label in enumerate(LABELS)
}

## SAVE LORA

In [ ]:
SAVE_PATH = "/kaggle/working/final_tweetbert"

os.makedirs(
    SAVE_PATH,
    exist_ok=True
)

In [ ]:
tokenizer.save_pretrained(SAVE_PATH)

merged_model.save_pretrained(SAVE_PATH)

print("Model saved!")
print(SAVE_PATH)

In [ ]:
!cd /kaggle/working && zip -r final_tweetbert.zip final_tweetbert

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/final_tweetbert.zip")

In [ ]:
!ls -lh /kaggle/working/final_tweetbert.zip